# Stage 4: The Benchmark. Co-folding vs Docking vs the Crystal Structure
**Project:** Can AI predict how drugs bind? Benchmarking AI structure and co-folding models on COX enzymes.

This is the head-to-head. For each drug-enzyme pair we compare three things:
- **Crystal structure** = the experimental answer key
- **AutoDock Vina** = classical docking, run into the crystal enzyme with the box centered on the real drug's site (so docking gets the pocket location for free)
- **Boltz** = AI co-folding from Stage 3, which got only sequence + SMILES

**Questions**
1. Which method puts the drug closer to the crystal pose? (ligand RMSD, success = under 2 Å)
2. Does docking find a good pose but score it badly? (top-1 pose vs best of 9)
3. Do the two independent methods agree with each other, including for aspirin, which has no crystal ligand here?
4. Do they get the **selectivity-relevant contacts** right: key residues (Arg120, Tyr355, Tyr385, Ser530), the COX-2 side pocket (513, 523), and the distance to Ser530?

**You need the Stage 3 results.** Either run this in the same session as Stage 3, or have `day3_outputs.zip` ready to upload (Colab: Files panel → upload; Kaggle: Add Data → Upload). Colab or Kaggle both work.

## 1. Setup

In [ ]:
import os, sys, glob, shutil, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")
print("Working folder:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython", "rdkit", "pandas", "matplotlib", "requests", "py3Dmol"])
try:
    from vina import Vina
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vina"])
try:
    from vina import Vina
    HAVE_VINA = True
except Exception as e:
    HAVE_VINA = False
    print("Vina is not available:", e)
if shutil.which("obabel") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openbabel", shell=True)
HAVE_OBABEL = shutil.which("obabel") is not None
print("Vina:", HAVE_VINA, "| OpenBabel:", HAVE_OBABEL)

**If Vina or OpenBabel shows `False`:** on Colab you can use the route that worked before (`!pip install -q condacolab`, `import condacolab; condacolab.install()`, then after the restart `!mamba install -q -y -c conda-forge vina openbabel rdkit`). The notebook still runs the Boltz-only parts if docking is unavailable.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests, py3Dmol
from Bio import Align
from Bio.PDB import PDBParser
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign

THREE2ONE = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G","HIS":"H",
             "ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S","THR":"T","TRP":"W",
             "TYR":"Y","VAL":"V","MSE":"M"}
SKIP = {"HOH","HEM","NAG","BMA","MAN","FUC","GOL","SO4","EDO","BOG","NA","CA","CL"}

def xyz(ls):
    return np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in ls])

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[:300])
    return r

## 2. Load the crystal structures (answer key)

In [ ]:
def load_experimental(key, pdb_id, prefer):
    text = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
    lines = text.splitlines()
    groups = {}
    for l in lines:
        if l.startswith("HETATM") and l[16] in " A" and l[17:20].strip() not in SKIP:
            groups.setdefault((l[17:20].strip(), l[21]), []).append(l)
    choice = next((k for p in prefer for k in groups if k[0] == p), None) or max(groups, key=lambda k: len(groups[k]))
    lig_name, chain_id = choice
    lig = groups[choice]
    atoms = [l for l in lines if l.startswith("ATOM") and l[21] == chain_id and l[16] in " A"]
    open(f"exp_{key}.pdb", "w").write("\n".join(atoms) + "\nEND\n")
    open(f"exp_{key}_view.pdb", "w").write("\n".join(atoms + lig) + "\nEND\n")
    s = PDBParser(QUIET=True).get_structure(key, f"exp_{key}.pdb")
    res = [r for r in s[0][chain_id] if r.id[0] == " " and "CA" in r]
    return {"key": key, "pdb_id": pdb_id, "lig_name": lig_name, "lig": lig, "lig_xyz": xyz(lig),
            "lig_block": "\n".join(lig) + "\nEND\n", "res": res,
            "by_resseq": {r.id[1]: r for r in res}}

EXP = {"cox1": load_experimental("cox1", "1EQG", ["IBP"]),
       "cox2": load_experimental("cox2", "3LN1", ["CEL", "S58"])}
LIGANDS = {"celecoxib": "Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F",
           "ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
           "aspirin": "CC(=O)Oc1ccccc1C(=O)O"}
JOBS = [("cox2", "celecoxib"), ("cox1", "ibuprofen"), ("cox1", "aspirin"), ("cox2", "aspirin")]
REF_DRUG = {"cox2": "celecoxib", "cox1": "ibuprofen"}
for k, e in EXP.items():
    print(k, e["pdb_id"], "| crystal ligand", e["lig_name"], "| residues", len(e["res"]))

## 3. Collect the Boltz poses from Stage 3
Looks for `pred_<enzyme>_<drug>_aligned.pdb` (already superimposed on the crystal structure) in the working folder, inside any `day3_outputs.zip`, and in Kaggle inputs.

In [ ]:
def find_files(pattern):
    hits = []
    for root in [".", "/kaggle/input", "/content"]:
        hits += glob.glob(os.path.join(root, "**", pattern), recursive=True)
    return hits

for z in find_files("day3_outputs.zip"):
    shutil.unpack_archive(z, "stage3_poses")
BOLTZ = {}
for enz, drug in JOBS:
    hits = find_files(f"pred_{enz}_{drug}_aligned.pdb")
    if hits:
        BOLTZ[(enz, drug)] = hits[0]
    print(f"{enz}_{drug}:", hits[0] if hits else "NOT FOUND")
if not BOLTZ:
    print("\nNo Boltz poses found. Upload day3_outputs.zip (or run Stage 3 here first), then rerun this cell.")

## 4. Dock with AutoDock Vina
The receptor is the crystal protein; the box is centered on the crystal drug (24 Å cube). Because the pocket location is given, this is the *easy* version of docking. Each job produces 9 poses.

In [ ]:
def prep_receptor(key):
    sh(f"obabel exp_{key}.pdb -O receptor_{key}.pdbqt -xr -h --partialcharge gasteiger")

def prep_ligand(smiles, name):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=7)
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, f"{name}.mol")
    sh(f"obabel {name}.mol -O {name}.pdbqt --partialcharge gasteiger")
    return f"{name}.pdbqt"

VINA = {}
if HAVE_VINA and HAVE_OBABEL:
    for key in EXP:
        prep_receptor(key)
    for enz, drug in JOBS:
        name = f"{enz}_{drug}"
        v = Vina(sf_name="vina", seed=42)
        v.set_receptor(f"receptor_{enz}.pdbqt")
        v.set_ligand_from_file(prep_ligand(LIGANDS[drug], f"lig_{drug}"))
        v.compute_vina_maps(center=EXP[enz]["lig_xyz"].mean(axis=0).tolist(), box_size=[24, 24, 24])
        v.dock(exhaustiveness=16, n_poses=9)
        v.write_poses(f"vina_{name}_out.pdbqt", n_poses=9, overwrite=True)
        scores = [float(x) for x in v.energies(n_poses=9)[:, 0]]
        blocks = []
        for i in range(1, len(scores) + 1):
            sh(f"obabel vina_{name}_out.pdbqt -O vina_{name}_pose{i}.pdb -f {i} -l {i}")
            blocks.append(open(f"vina_{name}_pose{i}.pdb").read())
        VINA[(enz, drug)] = {"scores": scores, "blocks": blocks}
        print(f"{name}: best Vina score {scores[0]:.2f} kcal/mol, {len(blocks)} poses")
else:
    print("Docking skipped (Vina/OpenBabel unavailable). Boltz-only analysis will still run.")

## 5. Measure everything the same way
Ligand RMSD uses RDKit with matching atoms (symmetry-aware) and no re-alignment, since all poses are already in the crystal frame. Contacts always use the crystal protein, so the three methods are judged on equal terms.

In [ ]:
def to_mol(block, smiles):
    return AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smiles), Chem.MolFromPDBBlock(block, removeHs=True))

def rmsd(block_a, block_b, smiles):
    try:
        return float(rdMolAlign.CalcRMS(to_mol(block_a, smiles), to_mol(block_b, smiles)))
    except Exception as e:
        print("RMSD failed:", str(e)[:120]); return None

def boltz_block(path):
    ls = [l for l in open(path) if l.startswith("HETATM") and l[17:20].strip() != "HOH"]
    chains = {l[21] for l in ls}
    chosen = "B" if "B" in chains else sorted(chains)[0]
    return "".join(l for l in ls if l[21] == chosen) + "END\n"

def block_xyz(block):
    ls = [l for l in block.splitlines() if l.startswith(("ATOM", "HETATM")) and l[76:78].strip().upper() != "H"
          and not l[12:16].strip().startswith("H")]
    return xyz(ls)

KEY = [("ARG", 120), ("TYR", 355), ("TYR", 385), ("SER", 530)]

def analyse_pose(enz, block):
    info = EXP[enz]
    lx = block_xyz(block)
    atoms = [(r, a) for r in info["res"] for a in r if not a.get_id().startswith("H")]
    pxyz = np.array([a.coord for _, a in atoms])
    d = np.linalg.norm(pxyz[:, None, :] - lx[None, :, :], axis=2).min(axis=1)
    touching = {(r.get_resname(), r.id[1]) for (r, _), dd in zip(atoms, d) if dd < 4.0}
    keys = [f"{n}{i}" for n, i in KEY if (n, i) in touching]
    side = [f"{n}{i}" for n, i in sorted(touching, key=lambda x: x[1]) if i in (513, 523)]
    og = info["by_resseq"].get(530)
    og_d = float(np.linalg.norm(lx - og["OG"].coord, axis=1).min()) if og is not None and "OG" in og else None
    site = float(np.linalg.norm(lx.mean(axis=0) - info["lig_xyz"].mean(axis=0)))
    return {"Site distance (Å)": round(site, 2), "Key residues": ", ".join(keys) or "none",
            "Side pocket (513/523)": ", ".join(side) or "none",
            "Ser530 OG (Å)": None if og_d is None else round(og_d, 2)}

In [ ]:
rows, best_of_n, agree = [], {}, {}
for enz, drug in JOBS:
    name = f"{enz}_{drug}"
    smi = LIGANDS[drug]
    is_ref = REF_DRUG[enz] == drug
    crystal = EXP[enz]["lig_block"]
    if is_ref:
        rows.append({"Job": name, "Method": "Crystal", "Ligand RMSD (Å)": 0.0, **analyse_pose(enz, crystal)})
    if (enz, drug) in VINA:
        vb = VINA[(enz, drug)]["blocks"]
        top = rmsd(vb[0], crystal, smi) if is_ref else None
        rr = [rmsd(b, crystal, smi) for b in vb] if is_ref else []
        rr = [x for x in rr if x is not None]
        best_of_n[name] = min(rr) if rr else None
        rows.append({"Job": name, "Method": "Vina top-1", "Ligand RMSD (Å)": None if top is None else round(top, 2),
                     "Vina score": round(VINA[(enz, drug)]["scores"][0], 2), **analyse_pose(enz, vb[0])})
    if (enz, drug) in BOLTZ:
        bb = boltz_block(BOLTZ[(enz, drug)])
        br = rmsd(bb, crystal, smi) if is_ref else None
        rows.append({"Job": name, "Method": "Boltz", "Ligand RMSD (Å)": None if br is None else round(br, 2),
                     **analyse_pose(enz, bb)})
        if (enz, drug) in VINA:
            agree[name] = rmsd(bb, VINA[(enz, drug)]["blocks"][0], smi)

table = pd.DataFrame(rows)
table.to_csv("day4_summary.csv", index=False)
table

**How to read it**
- **Ligand RMSD under 2 Å** = correct pose. Crystal rows are 0 by definition.
- **Site distance** = how far the drug's center is from the crystal drug's center.
- **Key residues / Side pocket** = which important residues the pose touches (4 Å). Compare each method with the Crystal row.
- **Ser530 OG** = closest approach to the residue aspirin acetylates.

In [ ]:
print("Best-of-9 Vina RMSD (any pose, shows sampling quality):")
for k, v in best_of_n.items():
    if v is not None:
        print(f"  {k}: {v:.2f} Å")
print("\nVina top-1 vs Boltz agreement (ligand RMSD between the two methods):")
for k, v in agree.items():
    print(f"  {k}: {'n/a' if v is None else f'{v:.2f} Å'}")

## 6. Plots

In [ ]:
ref = table[(table.Method != "Crystal") & table["Ligand RMSD (Å)"].notna()]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
jobs_ref = sorted(set(ref.Job))
x = np.arange(len(jobs_ref)); w = 0.26
def val(job, method):
    s = ref[(ref.Job == job) & (ref.Method == method)]["Ligand RMSD (Å)"]
    return float(s.iloc[0]) if len(s) else np.nan
ax[0].bar(x - w, [val(j, "Vina top-1") for j in jobs_ref], w, label="Vina top-1", color="#4C78A8")
ax[0].bar(x, [best_of_n.get(j) if best_of_n.get(j) is not None else np.nan for j in jobs_ref], w, label="Vina best of 9", color="#9ECAE9")
ax[0].bar(x + w, [val(j, "Boltz") for j in jobs_ref], w, label="Boltz", color="#E45756")
ax[0].axhline(2, ls="--", color="gray"); ax[0].set_xticks(x); ax[0].set_xticklabels(jobs_ref)
ax[0].set_title("Ligand RMSD vs crystal (Å); dashed = 2 Å"); ax[0].legend(fontsize=8)
ag = {k: v for k, v in agree.items() if v is not None}
ax[1].bar(list(ag), list(ag.values()), color="#72B7B2")
ax[1].axhline(2, ls="--", color="gray"); ax[1].set_title("Vina vs Boltz agreement (Å)")
ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig("day4_benchmark.png", dpi=150); plt.show()

## 7. 3D overlay: celecoxib in COX-2
Magenta: crystal drug · cyan: Vina pose · green: Boltz pose · orange sticks: residues 513 and 523.

In [ ]:
def show(enz, drug):
    v = py3Dmol.view(width=600, height=480)
    v.addModel(open(f"exp_{enz}_view.pdb").read(), "pdb")
    v.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.6}})
    v.setStyle({"model": 0, "hetflag": True}, {"stick": {"colorscheme": "magentaCarbon"}})
    v.addStyle({"model": 0, "resi": [513, 523]}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.2}})
    n = 1
    if (enz, drug) in VINA:
        v.addModel(VINA[(enz, drug)]["blocks"][0], "pdb")
        v.setStyle({"model": n}, {"stick": {"colorscheme": "cyanCarbon"}}); n += 1
    if (enz, drug) in BOLTZ:
        v.addModel(boltz_block(BOLTZ[(enz, drug)]), "pdb")
        v.setStyle({"model": n}, {"stick": {"colorscheme": "greenCarbon"}})
    v.zoomTo({"model": 0, "hetflag": True})
    return v

show("cox2", "celecoxib").show()

In [ ]:
show("cox1", "ibuprofen").show()

## 8. Summary for your README

In [ ]:
cols = list(table.columns)
print("| " + " | ".join(cols) + " |")
print("|" + "---|" * len(cols))
for _, r in table.iterrows():
    print("| " + " | ".join("n/a" if pd.isna(r[c]) else str(r[c]) for c in cols) + " |")

In [ ]:
shutil.rmtree("day4_outputs", ignore_errors=True)
os.makedirs("day4_outputs")
for f in ["day4_summary.csv", "day4_benchmark.png"]:
    if os.path.exists(f):
        shutil.copy(f, "day4_outputs")
shutil.make_archive("day4_outputs", "zip", "day4_outputs")
print("Download day4_outputs.zip (Kaggle: Output tab; Colab: Files panel)")

## What to write in your README
- **Accuracy:** success at 2 Å for Vina top-1, Vina best-of-9, and Boltz. With only two reference jobs, call it a case study, not a statistic.
- **Scoring vs sampling:** is Vina's best-of-9 much better than its top-1? If yes, docking *finds* the pose but *ranks* it badly.
- **Agreement:** where Vina and Boltz agree, is that also where they match the crystal? What do they say about aspirin?
- **Selectivity:** did each method put celecoxib's sulfonamide region near 513/523 in COX-2, like the crystal structure?
- **Mechanism:** how close did aspirin get to Ser530?

## Limitations (keep these in the README)
- **Unequal difficulty:** Vina was told where the pocket is; Boltz was not. Blind docking over the whole enzyme would be a fairer, harder test.
- **Training overlap:** Boltz was probably trained on these PDB entries, so its accuracy may be memory. Vina cannot memorize.
- Two reference complexes, one run each, one random seed. Results can change between runs.
- Aspirin has no crystal ligand here; it is judged by agreement, site, and contacts only.
- Rigid receptor docking, no heme or water; aspirin's covalent step is not modeled.

## Ideas for a Stage 5 / extension
- Repeat with a COX structure released **after** the model's training cutoff (the cleanest test of true prediction)
- Blind docking with a whole-protein box
- More seeds, more drugs, and PoseBusters-style physical checks